# RAGAS
To evaluate answer quality, I am going to generate question answer pairs using openai api


In [1]:
# Cell 1: Imports and setup
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import random
random.seed(69)



In [2]:
load_dotenv()

PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"

In [3]:
# Loads  chunks as LangChain documents
# Using the already-chunked data rather than raw PDFs since RAGAS should generate questions from what the pipeline actually retrieves



embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma(
    persist_directory=str(DATA_DIR / "chromadb"),
    collection_name="recursive_100",#typo of 1000
    embedding_function=embedding_model
)

# Get all documents from the collection
all_docs = vectorstore.get()
print(f"Loaded {len(all_docs['documents'])} chunks")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2573.16it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 31741 chunks


In [4]:
#df.head()

# Formal RAGAS Evaluation (Baseline vs. Hybrid)
This section runs the official quantitative comparison between the baseline (vector-only) and hybrid pipelines.
It tests both the General testset and the Multi-hop testset separately.

In [5]:
import os
import sys

import pandas as pd
from datasets import Dataset
from openai import OpenAI

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.config import TOP_K_RETRIEVAL
from src.generation.generate import build_context, generate_answer

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

ModuleNotFoundError: No module named 'config'

In [ ]:
# Hybrid Database Connections
from neo4j import GraphDatabase
from config import NEO4J_URI, NEO4J_USER
AUTH = (NEO4J_USER, "graphrag")
driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH)

def expand_from_chunks(chunk_ids, driver, max_triplets=40):
    cypher = """
    MATCH (c:Chunk) WHERE c.chunk_id IN $chunk_ids
    MATCH (c)-[:MENTIONS]->(entity)
    
    // Degree penalty graph hub filter (from hybrid notebook)
    WITH entity
    WHERE size([(entity)-[]-() | 1]) < 100
    
    MATCH (entity)-[r]->(neighbor)
    WHERE type(r) IN ['FOUND_IN','PRODUCES','STUDIED_WITH',
                      'IDENTIFIED_BY','BELONGS_TO','AFFECTS','CONTAINS']
      AND r.confidence >= 0.7
    RETURN DISTINCT
        entity.name  AS subject,
        type(r)      AS predicate,
        neighbor.name AS object,
        r.confidence AS confidence
    ORDER BY r.confidence DESC
    LIMIT $max_triplets
    """
    with driver.session() as session:
        result = session.run(cypher, chunk_ids=chunk_ids, max_triplets=max_triplets)
        return [(r["subject"], r["predicate"], r["object"], r["confidence"])
                for r in result]

In [ ]:
from ragas.llms import llm_factory
from langchain_openai import ChatOpenAI

# Setup DeepSeek as the Judge
deepseek_judge = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    max_tokens=8192,
    temperature=0.0
)
evaluator_llm = llm_factory(deepseek_judge)

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

In [ ]:
from tqdm.auto import tqdm

def run_eval_dataset(dataset_path, pipeline_type):
    print(f"\n== Evaluating {pipeline_type.upper()} on {Path(dataset_path).name} ==")
    testset_df = pd.read_csv(dataset_path)
    data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    
    retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K_RETRIEVAL})
    
    for i, row in tqdm(testset_df.iterrows(), total=len(testset_df)):
        query = row["user_input"]
        ground_truth = str(row.get("reference", ""))
        
        # Retrieval
        entry_docs = retriever.invoke(query)
        entry_chunk_ids = [d.metadata.get("chunk_id") for d in entry_docs]
        vector_context, contexts_list = build_context([(1.0, doc) for doc in entry_docs])
        
        if pipeline_type == "baseline":
            final_context = vector_context
        else:
            triplets = expand_from_chunks(entry_chunk_ids, driver)
            triplet_lines = [f"- {s} {p} {o} (confidence: {c:.2f})" for s, p, o, c in triplets]
            final_context = vector_context + "\n\nRelated knowledge graph facts:\n" + "\n".join(triplet_lines)
            if triplet_lines:
                contexts_list.append("Related knowledge graph facts:\n" + "\n".join(triplet_lines))
            
        answer = generate_answer(query, final_context, openai_client)
        
        data["question"].append(query)
        data["answer"].append(answer)
        data["contexts"].append(contexts_list)
        data["ground_truth"].append(ground_truth)
        
    return Dataset.from_dict(data)

In [ ]:
from ragas import evaluate

general_path = PROJECT_ROOT / "outputs" / "ragas_testset.csv"

# --- 1. Baseline on General --- 
 ds = run_eval_dataset(general_path, "baseline")
 result = evaluate(dataset=ds, metrics=metrics, llm=evaluator_llm)
 result.to_pandas().to_csv(PROJECT_ROOT / "outputs" / "baseline_eval_general.csv", index=False)
 print(result)

# --- 2. Hybrid on General ---
 ds = run_eval_dataset(general_path, "hybrid")
 result = evaluate(dataset=ds, metrics=metrics, llm=evaluator_llm)
 result.to_pandas().to_csv(PROJECT_ROOT / "outputs" / "hybrid_eval_general.csv", index=False)
 print(result)